# Case 4: Longitudinal follow-up, Mr. P. at the fifth annual visit

*Hands-On Electroretinography in the Age of AI*, Chapter 19, §19.4 (read with Chapter 18 §18.3 to §18.5).

Mr. P., 44, has retinitis pigmentosa and five annual visits. This notebook reproduces the version stamp check, the visit-pair flags of Table 19.5 against the Chapter 15 coefficient of repeatability (CoR), the observed annual change compared with published rates, and the fitted slope with its prediction interval for the next visit.

The values are constructed for teaching; they are not patient data.

**Setup.** In Google Colab, the first cell clones the repository. Elsewhere, run the notebook from the repository.

In [1]:
import os, sys
if 'google.colab' in sys.modules and not os.path.exists('/content/ERG-Analysis-API'):
    !git clone -q https://github.com/AI-Fairness-com/ERG-Analysis-API.git /content/ERG-Analysis-API
sys.path.insert(0, '/content/ERG-Analysis-API/examples')
sys.path.insert(0, os.path.abspath('examples') if os.path.isdir('examples') else os.getcwd())
from case_helpers import zone_table, check, REPORT

**Table 19.5 values.**

In [2]:
import numpy as np
import pandas as pd
visits = pd.DataFrame({
    'visit': [1, 2, 3, 4, 5],
    'year': [0, 1, 2, 3, 4],
    'la30_amp_uv': [34.0, 31.6, 25.4, 24.3, 20.2],
    'da3_b_amp_uv': [94.0, 85.7, 69.0, 51.8, 41.0],
})
visits

,visit,year,la30_amp_uv,da3_b_amp_uv
0,1,0,34.0,94.0
1,2,1,31.6,85.7
2,3,2,25.4,69.0
3,4,3,24.3,51.8
4,5,4,20.2,41.0


**Version stamp check (§19.4.1).** The four earlier audit records carry the same version stamp as the current report, so every comparison returns VERSION_IDENTICAL and the progression summary carries no caveat. The comparison function is the one of Chapter 17 §17.3.3.

In [3]:
import contextlib, importlib.util, io
spec = importlib.util.spec_from_file_location('chapter_17', 'chapters/ch17/chapter_17.py')
ch17 = importlib.util.module_from_spec(spec)
with contextlib.redirect_stdout(io.StringIO()):
    spec.loader.exec_module(ch17)
stamp = {'api_version': '1.0.0', 'model_weights_hash': 'a1b2',
         'config_hash': 'c3d4', 'normative_ref_version': 'Baker2025_v1'}
for v in visits['visit'][:-1]:
    print(f'Visit {v}:', ch17.compare_version_stamps(stamp, dict(stamp))['flag'])

Visit 1: VERSION_IDENTICAL
Visit 2: VERSION_IDENTICAL
Visit 3: VERSION_IDENTICAL
Visit 4: VERSION_IDENTICAL


**Visit-pair flags (§19.4.2).** Each DA 3.0 b-wave amplitude is compared with the previous visit on the log10 scale, against the CoR of 0.216 log µV (Table 15.3): a fall of 39% or a rise of 64%. All four changes stay inside the CoR, so all are NO_CHANGE. The LA 30 Hz implicit time rose by 0.9 ms between visits 4 and 5, against a CoR of 3.4 ms.

In [4]:
COR_LOG = 0.216
change = np.log10(visits['da3_b_amp_uv']).diff()
visits['b_change_pct'] = ((10 ** change - 1) * 100).round(1)
visits['flag'] = np.where(change.isna(), 'n/a',
                 np.where(change.abs() < COR_LOG, 'NO_CHANGE',
                 np.where(change < 0, 'SIGNIFICANT_PROGRESSION',
                          'SIGNIFICANT_IMPROVEMENT')))
print(visits[['visit', 'year', 'da3_b_amp_uv', 'b_change_pct', 'flag']].to_string(index=False))
print(f'\nCoR bounds: fall of {100 * (1 - 10 ** -COR_LOG):.0f}%, rise of {100 * (10 ** COR_LOG - 1):.0f}%')
print('LA 30 Hz implicit time, +0.9 ms against a CoR of 3.4 ms:',
      'NO_CHANGE' if abs(0.9) < 3.4 else 'FLAGGED')
assert list(visits['b_change_pct'][1:]) == [-8.8, -19.5, -24.9, -20.8]

 visit  year  da3_b_amp_uv  b_change_pct      flag
     1     0          94.0           NaN       n/a
     2     1          85.7          -8.8 NO_CHANGE
     3     2          69.0         -19.5 NO_CHANGE
     4     3          51.8         -24.9 NO_CHANGE
     5     4          41.0         -20.8 NO_CHANGE

CoR bounds: fall of 39%, rise of 64%
LA 30 Hz implicit time, +0.9 ms against a CoR of 3.4 ms: NO_CHANGE


**Observed rate and published rates (§19.4.3).** The LA 30 Hz amplitude fell from 34.0 to 20.2 µV over four years. The published means give context and are not a threshold.

In [5]:
first, last = visits['la30_amp_uv'].iloc[[0, -1]]
years = visits['year'].iloc[-1] - visits['year'].iloc[0]
rate = ((last / first) ** (1 / years) - 1) * 100
print(f'Observed change: {rate:.1f}% per year')
print('Published means: -10.0% per year (Berson, 2007); -13.2% per year in USH2A disease (Sandberg et al., 2008)')

Observed change: -12.2% per year
Published means: -10.0% per year (Berson, 2007); -13.2% per year in USH2A disease (Sandberg et al., 2008)


**Fitted slope and next-visit prediction (§19.4.4).** The model works on the log LA 30 Hz amplitude. With one patient's five visits there is no cohort to share information with, so the fit reduces to a straight line through his own visits. Its intervals use the t distribution with 3 degrees of freedom; these give the figures printed in the book: a slope of −12.2% per year (95% CI −16.1% to −8.2%) and a 95% prediction interval of 14.7 to 22.1 µV for the next visit. With a cohort, the mixed-effects model of Chapter 18 (`chapters/ch18/longitudinal_lme.py`) is used instead.

In [6]:
import statsmodels.api as sm
t = visits['year'].to_numpy(float)
fit = sm.OLS(np.log(visits['la30_amp_uv']), sm.add_constant(t)).fit()
pct = lambda v: 100 * (np.exp(v) - 1)
lo, hi = fit.conf_int().iloc[1]
print(f'Slope: {pct(fit.params.iloc[1]):.1f}% per year (95% CI {pct(lo):.1f}% to {pct(hi):.1f}%)')
nxt = fit.get_prediction(np.array([[1.0, 5.0]])).summary_frame(alpha=0.05)
print(f"Next visit: 95% prediction interval {np.exp(nxt['obs_ci_lower'].iloc[0]):.1f} "
      f"to {np.exp(nxt['obs_ci_upper'].iloc[0]):.1f} µV")

Slope: -12.2% per year (95% CI -16.1% to -8.2%)
Next visit: 95% prediction interval 14.7 to 22.1 µV


**The trajectory.** Visits on a log scale, the fitted line and the next-visit prediction interval.

In [7]:
import matplotlib.pyplot as plt
grid = np.linspace(0, 5, 50)
pred = fit.get_prediction(sm.add_constant(grid)).summary_frame(alpha=0.05)
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(t, visits['la30_amp_uv'], 'o', color='#16213e', label='Visits')
ax.plot(grid, np.exp(pred['mean']), color='#e76f51', label='Fitted slope')
ax.errorbar(5, np.exp(nxt['mean'].iloc[0]),
            yerr=[[np.exp(nxt['mean'].iloc[0]) - np.exp(nxt['obs_ci_lower'].iloc[0])],
                  [np.exp(nxt['obs_ci_upper'].iloc[0]) - np.exp(nxt['mean'].iloc[0])]],
            fmt='s', color='#4ecdc4', capsize=4, label='Next visit, 95% PI')
ax.set_yscale('log')
ax.set_xlabel('Years from the first visit')
ax.set_ylabel('LA 30 Hz amplitude (µV)')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()